In [2]:
from pymatgen.ext.optimade import OptimadeRester
from pymatgen.analysis.dimensionality import get_dimensionality_gorai
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer 

opt = OptimadeRester(["alexandria"], timeout=None)

pt_data = opt.get_structures_with_filter('(elements HAS "Pt") AND (nelements=1)')
all_pt = pt_data["alexandria"]   


pt_structures = []
for structure in all_pt.values():
    if len(structure) >= 25:
        continue
    if get_dimensionality_gorai(structure, max_hkl=2) != 3:
        continue

    conventional_structure = SpacegroupAnalyzer(
        structure,
        symprec=0.01
    ).get_conventional_standard_structure()
    pt_structures.append(conventional_structure)

print(len(pt_structures))

alexandria: 100%|██████████| 23/23 [00:00<00:00, 90.46it/s]


20


In [3]:
import json
from atomate2.forcefields.flows.elastic import ElasticMaker
from jobflow import run_locally
from pathlib import Path

run_directory = Path("pt_elastic_runs")
run_directory.mkdir(parents=True, exist_ok=True)

elastic_maker = ElasticMaker.from_force_field_name(
    force_field_name="MatterSim",
    relax_initial_structure=True,
    sym_reduce=True,
)


pt_results = []

for i, structure in enumerate(pt_structures):

    responses = run_locally(
        elastic_maker.make(structure),
        create_folders=True,
        root_dir=run_directory / f"Pt_{i}",
        ensure_success=True,
    )

    elastic_document = None

    for job_responses in responses.values():
        for response in job_responses.values():

            output = response.output

            if (
                hasattr(output, "derived_properties")
                and output.derived_properties is not None
            ):
                elastic_document = output

    if elastic_document is None:
        raise RuntimeError(
            f"No ElasticDocument found for Pt structure {i}."
        )

    properties = elastic_document.derived_properties

    pt_results.append({
        "structure_id": f"Pt_{i}",
        "structure": structure.as_dict(),
        "k_reuss": properties.k_reuss,
        "k_voigt": properties.k_voigt,
        "k_vrh": properties.k_vrh,
    })


with open("pt_elastic_results.json", "w") as file:
    json.dump(pt_results, file, indent=2)

2026-09-08 10:12:13,683 INFO Started executing jobs locally
2026-09-08 10:12:13,691 INFO Starting job - Force field relax (908bdeda-a7eb-4413-b9a3-c18e0d055fb5)


/tmp/ipykernel_2290181/395113681.py:9: UserWarning: Fixed symmetry relaxations are automatically enabled to improve elastic tensor stability. To disable this specify ForceFieldRelaxMaker objects explicitly. 
  elastic_maker = ElasticMaker.from_force_field_name(


2026-09-08 10:12:23.101 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:12:37,302 INFO Finished job - Force field relax (908bdeda-a7eb-4413-b9a3-c18e0d055fb5)
2026-09-08 10:12:37,304 INFO Starting job - generate_elastic_deformations (90a308ba-7924-4fc1-8860-7d52b46c7938)


/home/ri52yof/projects/Bulk_Modulus_Project/.venv/lib/python3.12/site-packages/atomate2/ase/utils.py:481: UserWarning: Only FixAtoms and FixCartesian is supported by Pymatgen. Other constraints will not be set.
  struct = self.ase_adaptor.get_structure(


2026-09-08 10:12:38,132 INFO Finished job - generate_elastic_deformations (90a308ba-7924-4fc1-8860-7d52b46c7938)
2026-09-08 10:12:38,134 INFO Starting job - run_elastic_deformations (53a8aae6-efcd-41df-b1ff-3174fb812c34)
2026-09-08 10:12:38,182 INFO Finished job - run_elastic_deformations (53a8aae6-efcd-41df-b1ff-3174fb812c34)
2026-09-08 10:12:38,188 INFO Starting job - Force field relax 1/18 (f65458fa-c734-4b56-9c35-b26e3e88983f)
2026-09-08 10:12:38.189 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:12:39,046 INFO Finished job - Force field relax 1/18 (f65458fa-c734-4b56-9c35-b26e3e88983f)
2026-09-08 10:12:39,047 INFO Starting job - Force field relax 2/18 (e334b026-1f7a-4274-b0d3-c24baee68417)
2026-09-08 10:12:39.049 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model


...

2026-09-08 10:25:08,771 INFO Finished job - Force field relax 2/20 (05b9e30d-9398-4f7c-a9be-925ae91d563f)
2026-09-08 10:25:08,773 INFO Starting job - Force field relax 3/20 (261f9581-b722-4e78-a184-1e43f564d4c9)
2026-09-08 10:25:08.774 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:25:11,902 INFO Finished job - Force field relax 3/20 (261f9581-b722-4e78-a184-1e43f564d4c9)
2026-09-08 10:25:11,905 INFO Starting job - Force field relax 4/20 (76c1bdd9-113e-4e8d-957b-a545ca18d2e5)
2026-09-08 10:25:11.907 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:25:14,720 INFO Finished job - Force field relax 4/20 (76c1bdd9-113e-4e8d-957b-a545ca18d2e5)
2026-09-08 10:25:14,723 INFO Starting job - Force field relax 5/20 (ed2eae94-cca0-45a0-8bd0-6987623eabb0)
2026-09-08 10:25:14.724 | INFO     | mattersim.forcefield.potential:from_che

/home/ri52yof/projects/Bulk_Modulus_Project/.venv/lib/python3.12/site-packages/atomate2/ase/utils.py:481: UserWarning: Only FixAtoms and FixCartesian is supported by Pymatgen. Other constraints will not be set.
  struct = self.ase_adaptor.get_structure(


2026-09-08 10:26:25,434 INFO Finished job - Force field relax (9cc6d37e-90c2-459d-a52c-fbfcf37665b5)
2026-09-08 10:26:25,438 INFO Starting job - generate_elastic_deformations (e7ad2728-a95e-4bfc-ab0b-29d6ea0ac145)
2026-09-08 10:26:26,255 INFO Finished job - generate_elastic_deformations (e7ad2728-a95e-4bfc-ab0b-29d6ea0ac145)
2026-09-08 10:26:26,257 INFO Starting job - run_elastic_deformations (fbe55120-463a-400f-aba6-8dca82ad70bd)
2026-09-08 10:26:27,115 INFO Finished job - run_elastic_deformations (fbe55120-463a-400f-aba6-8dca82ad70bd)
2026-09-08 10:26:27,130 INFO Starting job - Force field relax 1/18 (5fe92d22-045d-4913-853e-66b7bf020d4a)
2026-09-08 10:26:27.132 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model


/home/ri52yof/projects/Bulk_Modulus_Project/.venv/lib/python3.12/site-packages/atomate2/ase/utils.py:481: UserWarning: Only FixAtoms and FixCartesian is supported by Pymatgen. Other constraints will not be set.
  struct = self.ase_adaptor.get_structure(


2026-09-08 10:26:31,321 INFO Finished job - Force field relax 1/18 (5fe92d22-045d-4913-853e-66b7bf020d4a)
2026-09-08 10:26:31,324 INFO Starting job - Force field relax 2/18 (d739213e-ba39-4e2f-8eb9-2121bd00f160)
2026-09-08 10:26:31.326 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:26:36,563 INFO Finished job - Force field relax 2/18 (d739213e-ba39-4e2f-8eb9-2121bd00f160)
2026-09-08 10:26:36,565 INFO Starting job - Force field relax 3/18 (47a99b63-0089-4bcc-a180-8fd7bf4138aa)
2026-09-08 10:26:36.567 | INFO     | mattersim.forcefield.potential:from_checkpoint:922 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
2026-09-08 10:26:40,966 INFO Finished job - Force field relax 3/18 (47a99b63-0089-4bcc-a180-8fd7bf4138aa)
2026-09-08 10:26:40,969 INFO Starting job - Force field relax 4/18 (ffb49c74-6adb-4f87-b3f3-456177d28e57)
2026-09-08 10:26:40.970 | INFO     | mattersim.forcefield.potential:from_che

In [ ]:
import json
import math


# Load the supplied Cu and Cu-Pt data
with open("database.json", "r") as file:
    original_data = json.load(file)


# Load the newly calculated Pt data
with open("omitted_20Pt_strucs.json", "r") as file:
    pt_data = json.load(file)


# Combine Cu, Cu-Pt and Pt
combined_data = original_data + pt_data

clean_data = []
removed_data = []


for record in combined_data:

    try:
        k_reuss = float(record["k_reuss"])
        k_voigt = float(record["k_voigt"])
        k_vrh = float(record["k_vrh"])

        valid = (
            math.isfinite(k_reuss)
            and math.isfinite(k_voigt)
            and math.isfinite(k_vrh)
            and k_reuss > 0
            and k_voigt > 0
            and k_vrh > 0
            and k_voigt > k_reuss
        )

    except (KeyError, TypeError, ValueError):
        valid = False

    if valid:
        clean_data.append(record)
    else:
        removed_data.append(record)


# Save the cleaned database
with open("clean_database.json", "w") as file:
    json.dump(clean_data, file, indent=2)




TypeError: unsupported operand type(s) for +: 'dict' and 'list'